In [ ]:
from pathlib import Path
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.normalizers import BertNormalizer
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing
from transformers import PreTrainedTokenizerFast

DATA_FILE = "./data.txt"
TOKENIZER_DIR = Path("./model")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
VOCAB_SIZE = 30000
MIN_FREQUENCY = 2
MAX_LENGTH = 512
MAX_DOCS = 200_000
MIN_DOC_CHARS = 200

SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
DOC_SEP = "<|eos|>"

In [ ]:
def iter_text_docs(path, sep=DOC_SEP, min_doc_chars=MIN_DOC_CHARS, max_docs=MAX_DOCS):
    acc = []
    seen = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if line == sep:
                doc = "\n".join(acc).strip()
                acc = []

                if doc and len(doc) >= min_doc_chars:
                    yield doc
                    seen += 1
                    if max_docs is not None and seen >= max_docs:
                        return
            else:
                acc.append(line)

        if acc:
            doc = "\n".join(acc).strip()
            if doc and len(doc) >= min_doc_chars:
                yield doc

In [ ]:
def corpus_iterator(path, chunk_size=10_000):
    buffer = []

    for doc in iter_text_docs(path):
        buffer.append(doc)

        if len(buffer) >= chunk_size:
            yield buffer
            buffer = []

    if buffer:
        yield buffer

In [ ]:
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

tokenizer.normalizer = BertNormalizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=False,
    lowercase=False,   # True لو عايز uncased
)

tokenizer.pre_tokenizer = BertPreTokenizer()

trainer = WordPieceTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
    continuing_subword_prefix="##",
)

tokenizer.train_from_iterator(corpus_iterator(DATA_FILE), trainer=trainer)

tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", tokenizer.token_to_id("[CLS]")),
        ("[SEP]", tokenizer.token_to_id("[SEP]")),
    ],
)

tokenizer_json_path = TOKENIZER_DIR / "tokenizer.json"
tokenizer.save(str(tokenizer_json_path))

In [ ]:
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(tokenizer_json_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
    model_max_length=MAX_LENGTH,
)

hf_tokenizer.save_pretrained(TOKENIZER_DIR)

print("Saved tokenizer to:", TOKENIZER_DIR)
print("Vocab size:", hf_tokenizer.vocab_size)

In [ ]:
sample = "Transformers are useful for NLP tasks like classification and QA."
enc = hf_tokenizer(sample)
print(enc["input_ids"][:30])
print(hf_tokenizer.convert_ids_to_tokens(enc["input_ids"][:30]))
print(hf_tokenizer.decode(enc["input_ids"]))

In [ ]:
text = "unbelievable classification retrieval preprocessing"
tokens = hf_tokenizer.tokenize(text)
print(tokens)

In [ ]:
samples = [
    "This is a simple English sentence.",
    "BERT uses masked language modeling.",
    "Question answering and token classification are common tasks."
]

for s in samples:
    toks = hf_tokenizer.tokenize(s)
    unk_count = toks.count("[UNK]")
    print(s)
    print(toks)
    print("UNK ratio:", unk_count / max(1, len(toks)))
    print("-" * 50)